# TimeGuard — GPU Backend

Bu notebook Colab GPU runtime'ında FastAPI backend'i çalıştırır.

1. Runtime → Change runtime type → **T4 GPU** seç
2. Tüm hücreleri sırayla çalıştır
3. Son hücredeki ngrok URL'yi yerel Streamlit'e yapıştır

In [ ]:
# ── 1. GPU Kontrol + Kütüphane Kurulumu ──
!nvidia-smi

!pip install fastapi uvicorn mlflow "sqlalchemy<2.1" psycopg2-binary shap pydantic-settings pyngrok

# PyTorch GPU (Colab zaten yüklü, garanti olsun)
!pip install torch --index-url https://download.pytorch.org/whl/cu121

# cuML (GPU-accelerated sklearn)
!pip install cuml-cu12

In [ ]:
# ── 2. PostgreSQL Kurulumu ──
!apt-get update -qq && apt-get install -y -qq postgresql > /dev/null 2>&1
!service postgresql start

!sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE anomaly_detection OWNER postgres;"

print('PostgreSQL ready')

In [ ]:
# ── 3. Proje Kodunu Yükle ──
# Seçenek A: GitHub'dan clone
!git clone https://github.com/afdemir06/anomaly-deteciton.git anomaly-detection 2>/dev/null || true
%cd anomaly-detection

# Seçenek B: Dosya yükle (colab dosyası)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/anomaly-detection

# Mevcut dosya yapısını kontrol et
!ls -la

In [ ]:
# ── 4. .env Dosyasını Oluştur ──
%%writefile .env
# PostgreSQL
POSTGRES_HOST=localhost
POSTGRES_PORT=5432
POSTGRES_USER=postgres
POSTGRES_PASSWORD=postgres
POSTGRES_DB=anomaly_detection

# MLflow
MLFLOW_TRACKING_URI=http://localhost:5000
MLFLOW_EXPERIMENT_NAME=anomaly-detection

# Isolation Forest
IF_N_ESTIMATORS=100
IF_CONTAMINATION=0.05
IF_RANDOM_STATE=42

# DBSCAN
DBSCAN_EPS=0.5
DBSCAN_MIN_SAMPLES=5
DBSCAN_METRIC=euclidean

# LSTM Autoencoder
LSTM_SEQ_LENGTH=30
LSTM_HIDDEN_DIM=64
LSTM_NUM_EPOCHS=25
LSTM_LEARNING_RATE=0.001
LSTM_THRESHOLD_PERCENTILE=95.0

# Ensemble
ENSEMBLE_MIN_VOTES=2

# Label
LABEL_COLUMN=Class

# DBSCAN Subsample
DBSCAN_SUBSAMPLE_SIZE=10000

# SHAP
SHAP_MAX_DISPLAY=20
SHAP_SAMPLE_SIZE=100

# Docker Service URLs
API_HOST=0.0.0.0
API_PORT=8000
MLFLOW_HOST=localhost
MLFLOW_PORT=5000

In [ ]:
# ── 5. MLflow Başlat ──
import subprocess, time

mlflow_proc = subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:///mlflow.db",
     "--default-artifact-root", "./mlruns",
     "--host", "localhost",
     "--port", "5000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print(f'MLflow started (pid={mlflow_proc.pid})')

In [ ]:
# ── 6. ngrok ile Public URL Oluştur ──
from pyngrok import ngrok, conf

# ngrok token (https://dashboard.ngrok.com/signup)
NGROK_TOKEN = "SENIN_NGROK_TOKEN_BURAYA"  # ← BUNU DEĞİŞTİR

if NGROK_TOKEN == "SENIN_NGROK_TOKEN_BURAYA":
    print('⚠️  Lütfen NGROK_TOKEN değerini değiştir!')
    print('   https://dashboard.ngrok.com/get-started/your-authtoken')
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(8000)
    print(f'\n🔗 API URL: {tunnel.public_url}')
    print(f'   Bu URL\'yi Streamlit sidebar\'ına yapıştırın')

In [ ]:
# ── 7. FastAPI Backend Başlat ──
import subprocess, sys, time

print('Starting FastAPI server...')
print('GPU:', end=' ')
!python -c "import torch; print('CUDA available' if torch.cuda.is_available() else 'CPU only')"
print()

proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"]
)
time.sleep(3)
print(f'FastAPI started (pid={proc.pid})')
print('Check ngrok URL from previous cell')